In [1]:
import pandas as pd


In [4]:
train = pd.read_csv('train.csv')

In [5]:
test = pd.read_csv('test.csv')

In [8]:
print('missing values')
print(train.isnull().sum())

missing values
Index               0
geohash             0
day                 0
timestamp           0
demand              0
RoadType          600
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      2495
Weather           797
dtype: int64


In [9]:
train['RoadType'] = train['RoadType'].fillna('Unknown')
test['RoadType'] = test['RoadType'].fillna('Unknown')

In [10]:
train['Weather'] = train['Weather'].fillna('Unknown')
test['Weather'] = test['Weather'].fillna('Unknown')

In [11]:
train['Temperature'] = train.groupby('day')['Temperature'].transform(lambda x: x.fillna(x.median()))
test['Temperature'] = test.groupby('day')['Temperature'].transform(lambda x: x.fillna(x.median()))

In [12]:
print(train.isnull().sum())
print(test.isnull().sum())

Index            0
geohash          0
day              0
timestamp        0
demand           0
RoadType         0
NumberofLanes    0
LargeVehicles    0
Landmarks        0
Temperature      0
Weather          0
dtype: int64
Index            0
geohash          0
day              0
timestamp        0
RoadType         0
NumberofLanes    0
LargeVehicles    0
Landmarks        0
Temperature      0
Weather          0
dtype: int64


In [15]:
def feature_engineering(df):
  df[['Hour', 'Minute']] = df['timestamp'].str.split(':', expand=True).astype(int)
  df = df.drop(columns=['timestamp'])
  df['LargeVehicles'] = df['LargeVehicles'].map({'Allowed': 1, 'Not Allowed': 0})
  df['Landmarks'] = df['Landmarks'].map({'Yes': 1, 'No': 0})
  return df

In [17]:
train = feature_engineering(train)
test = feature_engineering(test)

In [18]:
train.head()

,Index,geohash,day,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,Hour,Minute
0,0,qp02z1,48,0.048804,Unknown,1,0,0,16.398024,Unknown,0,0
1,1,qp02zt,48,0.118507,Residential,3,1,1,31.104565,Sunny,0,0
2,2,qp08bj,48,0.027132,Residential,1,0,0,25.919267,Sunny,0,0
3,3,qp08gt,48,0.003272,Residential,1,0,0,16.398024,Rainy,0,0
4,4,qp02zq,48,0.010819,Residential,1,0,0,10.803667,Rainy,0,0


In [19]:
train.tail()

,Index,geohash,day,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,Hour,Minute
77294,77294,qp0d4n,49,0.067203,Residential,1,0,0,11.501664,Rainy,2,0
77295,77295,qp0d4q,49,0.022859,Residential,3,1,1,14.715254,Foggy,2,0
77296,77296,qp0d4w,49,0.141342,Residential,3,1,1,19.678860,Sunny,2,0
77297,77297,qp0dhw,49,0.087574,Residential,1,0,0,22.573958,Sunny,2,0
77298,77298,qp0djq,49,0.002944,Residential,3,1,1,1.322034,Snowy,2,0


In [21]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.7 MB/s eta 0:00:00


In [22]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split

In [26]:
X = train.drop(columns=['demand', 'Index']) # We drop 'demand' and 'Index' from the training data
y = train['demand']

In [27]:
test_features = test.drop(columns=['Index'])

In [28]:
cat_cols = ['geohash', 'RoadType', 'Weather'] # Tell CatBoost which columns are text/categorical

In [29]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42) #Split the train data (80% for training, 20% for testing our accuracy internally)

In [30]:
model = CatBoostRegressor(
    iterations=500,        # Number of decision trees
    learning_rate=0.1,     # How fast it learns
    depth=8,               # How deep the trees go
    eval_metric='RMSE',    # Root Mean Square Error (standard hackathon metric)
    cat_features=cat_cols, # Passing our text columns
    random_seed=42,
    verbose=50             # Print an update every 50 steps
)

In [31]:

model.fit(X_train, y_train, eval_set=(X_val, y_val)) #Train the Model!

0:	learn: 0.1317388	test: 0.1315850	best: 0.1315850 (0)	total: 158ms	remaining: 1m 18s
50:	learn: 0.0456242	test: 0.0450702	best: 0.0450702 (50)	total: 5.13s	remaining: 45.2s
100:	learn: 0.0411051	test: 0.0413898	best: 0.0413898 (100)	total: 8.76s	remaining: 34.6s
150:	learn: 0.0386205	test: 0.0394027	best: 0.0394027 (150)	total: 11.8s	remaining: 27.2s
200:	learn: 0.0373431	test: 0.0386044	best: 0.0386044 (200)	total: 14.6s	remaining: 21.7s
250:	learn: 0.0361004	test: 0.0378981	best: 0.0378981 (250)	total: 19.3s	remaining: 19.1s
300:	learn: 0.0348983	test: 0.0370640	best: 0.0370635 (299)	total: 22.5s	remaining: 14.9s
350:	learn: 0.0337688	test: 0.0362980	best: 0.0362980 (350)	total: 25.7s	remaining: 10.9s
400:	learn: 0.0328236	test: 0.0358150	best: 0.0358150 (400)	total: 30s	remaining: 7.41s
450:	learn: 0.0322373	test: 0.0355728	best: 0.0355728 (450)	total: 33.9s	remaining: 3.68s
499:	learn: 0.0317546	test: 0.0353411	best: 0.0353390 (498)	total: 37.1s	remaining: 0us

bestTest = 0.03533

CatBoostRegressor(cat_features=['geohash', 'RoadType', 'Weather'], depth=8, eval_metric='RMSE', iterations=500, learning_rate=0.1, loss_function='RMSE', random_seed=42, verbose=50)

In [32]:
predictions = model.predict(test_features) #Predict the 'demand' for the future test dataset

In [33]:
submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': predictions
}) #the final Submission of CSV

In [34]:
submission.to_csv('my_first_submission.csv', index=False)
print("\nSUCCESS! 'my_first_submission.csv' has been created.")


SUCCESS! 'my_first_submission.csv' has been created.
